In [186]:
import numpy as np
import scipy.signal as signal
import plotly.express as px
import pandas as pd # pour les dataframes
import sounddevice as sd # pour jouer les sons associés aux signaux

#  1. Analyse de l’évolution de la distance Terre-Lune

## 1.1 Chargement et affichage des données

In [187]:
data = np.load("data/distance_earth_moon_2011.npz")
distances = data['distances']

In [188]:
moy = np.mean(distances)
print ("La distance Terre-Lune moyenne vaut :", moy)

La distance Terre-Lune moyenne vaut : 385072.98540145985


In [189]:
K = np.arange(len(distances))
px.line(x = K, y = distances, labels={"x":"Numéro du jour", "y":"Distance Terre-Lune"}, title="Evolution de la distance Terre-Lune en fonction des jours").show()

## 1.2 Analyse fréquentielle

1/ Comme l'unité du signal est le jour, alors l'unité de l'axe fréquentiel sera le par jour c'est à dire le jour-1

In [190]:
K = len(distances)
distances_centrees = distances - np.mean(distances)
N_fft = 4 * K  # Zero-padding demandé (4K-points) 
FFT = np.fft.fft(distances_centrees, n=N_fft)
FFT_2 = FFT[:len(FFT)//2]
spectre = np.abs(FFT_2)
freq = np.arange(N_fft/2) / N_fft

px.line(x = freq, y = spectre, labels={"x":"Fréquence (en par jour)", "y":"Amplitude"}, title="Spectre du signal").show()

In [191]:
px.line(x = freq, y = 20 * np.log10(spectre), labels={"x":"Fréquence (en par jour)", "y":"Amplitude en dB"}, title="Spectre du signal en dB").show()

In [192]:
K = len(distances_centrees)
w_H = signal.windows.hann(K)
w_R = signal.windows.boxcar(K)
signal_fenetre_H = distances_centrees * w_H
signal_fenetre_R = distances_centrees * w_R

N_fft = 4 * K  # Zero-padding demandé (4K-points) 
spectre_H = np.abs(np.fft.fft(signal_fenetre_H, n=N_fft))
spectre_R = np.abs(np.fft.fft(signal_fenetre_R, n=N_fft))
freq = np.arange(N_fft) / N_fft

px.line(x = freq, y = spectre_H, labels={"x":"Fréquence (en par jour)", "y":"Amplitude"}, title="Spectre du signal fenetree hanning").show()
px.line(x = freq, y = np.log10(spectre_H), labels={"x":"Fréquence (en par jour)", "y":"Amplitude"}, title="Spectre du signal fenetree hanning en log10").show()
px.line(x = freq, y = spectre_R, labels={"x":"Fréquence (en par jour)", "y":"Amplitude"}, title="Spectre du signal fenetree rectangle").show()
px.line(x = freq, y = np.log10(spectre_R), labels={"x":"Fréquence (en par jour)", "y":"Amplitude"}, title="Spectre du signal fenetree rectangle en log 10").show()

# 2 Bureau d’étude analyse spectrale

# Signal 2

La finesse d'analyse dépend du lobe principal de la fenêtre choisie. Une fenêtre rectangulaire a le lobe le plus fin mais beaucoup de fuites. Une fenêtre de Hanning élargit le pic principal (moins bonne résolution) mais réduit les fuites (meilleure dynamique).
La résolution fréquentielle correspond à la capacité à distinguer deux fréquences proches. Elle est liée à la durée d'observation du signal. 

In [193]:
data = np.load('data/s2.npz')
t = data['t']
s = data['s']

In [194]:
dt = t[1] - t[0]
fe = 1 / dt
print(f"Fréquence d'échantillonnage fe = {fe} Hz")

px.line(x = t, y = s, labels={"x":"Temps (s)", "y":"Amplitude"}, title="Signal 2").show()

Fréquence d'échantillonnage fe = 40000.0 Hz


In [197]:
s_centre = s - np.mean(s) 

N = 512
s_analyse = s_centre[:N]

w_H = signal.windows.hann(N)
s_fenetre = s_analyse * w_H

# TFD sur 2048 points 
N_fft = 4 * N
spectre = np.abs(np.fft.fft(s_fenetre, n=N_fft))
freq = np.arange(N_fft)

px.line(x = freq, y = spectre, labels={"x":"Fréquence (Hz)", "y":"Amplitude"}, title="Spectre du signal 2").show()
px.line(x = freq, y = 20 * np.log10(spectre), labels={"x":"Fréquence (Hz)", "y":"Amplitude en dB"}, title="Spectre du signal 2 en log10").show()